In [1]:
%pip install -q dvc pyyaml

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadVision")

files_to_check = {
    "Dataset v1": DRIVE_ROOT / "datasets/rdd2022_subset_v1.zip",
    "Dataset v2": DRIVE_ROOT / "datasets/rdd2022_subset_v2.zip",
    "Model v1": DRIVE_ROOT / "models/v1/best.pt",
    "Model v2": DRIVE_ROOT / "models/v2/best.pt",
}

for name, path in files_to_check.items():
    print(name, "OK" if path.is_file() else "MISSING")

assert all(p.is_file() for p in files_to_check.values())

Mounted at /content/drive
Dataset v1 OK
Dataset v2 OK
Model v1 OK
Model v2 OK


In [4]:
from pathlib import Path
import subprocess

PROJECT = Path("/content/roadvision-mlops")
PROJECT.mkdir(parents=True, exist_ok=True)

def run(*args):
    subprocess.run(
        list(args),
        cwd=PROJECT,
        check=True
    )

if not (PROJECT / ".git").exists():
    run("git", "init", "-b", "main")

if not (PROJECT / ".dvc").exists():
    run("dvc", "init")

for folder in ["data", "models", "src", "reports"]:
    (PROJECT / folder).mkdir(exist_ok=True)

print("Project initialized:", PROJECT)

Project initialized: /content/roadvision-mlops


In [6]:
import shutil

copy_map = {
    DRIVE_ROOT / "datasets/rdd2022_subset_v1.zip":
        PROJECT / "data/rdd2022_v1.zip",

    DRIVE_ROOT / "datasets/rdd2022_subset_v2.zip":
        PROJECT / "data/rdd2022_v2.zip",

    DRIVE_ROOT / "models/v1/best.pt":
        PROJECT / "models/yolo11n_v1.pt",

    DRIVE_ROOT / "models/v2/best.pt":
        PROJECT / "models/yolo11n_v2.pt",
}

for source, destination in copy_map.items():
    if not destination.exists():
        shutil.copy2(source, destination)

    print(destination.name, round(destination.stat().st_size / 1024**2, 2), "MB")

rdd2022_v1.zip 115.54 MB
rdd2022_v2.zip 161.24 MB
yolo11n_v1.pt 5.25 MB
yolo11n_v2.pt 5.25 MB


In [7]:
run(
    "dvc", "add",
    "data/rdd2022_v1.zip",
    "data/rdd2022_v2.zip",
    "models/yolo11n_v1.pt",
    "models/yolo11n_v2.pt"
)

run("dvc", "status")

In [8]:
REMOTE = DRIVE_ROOT / "dvc-storage"
REMOTE.mkdir(parents=True, exist_ok=True)

run(
    "dvc", "remote", "add",
    "-d", "drive-storage",
    str(REMOTE)
)

run("dvc", "push")
print("DVC data pushed to Drive!")

DVC data pushed to Drive!


In [9]:
run("git", "add", ".")
run(
    "git",
    "-c", "user.name=RoadVision",
    "-c", "user.email=roadvision@example.invalid",
    "commit",
    "-m", "Track RoadVision v1 and v2 datasets and models with DVC"
)

run("git", "status", "--short")

In [10]:

script = r'''
import zipfile
import hashlib
import json
from pathlib import Path
from collections import Counter

ROOT = Path(__file__).resolve().parents[1]

def read_dataset(path):
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        splits = {}
        eval_hashes = {}

        for split in ["train", "val", "test"]:
            images = sorted(
                n for n in names
                if n.startswith(f"images/{split}/")
                and n.lower().endswith(
                    (".jpg", ".jpeg", ".png")
                )
            )

            boxes = Counter()
            backgrounds = 0

            for image_name in images:
                stem = Path(image_name).stem
                label_name = f"labels/{split}/{stem}.txt"

                if label_name not in names:
                    raise ValueError(
                        f"Missing label: {label_name}"
                    )

                lines = z.read(label_name).decode().splitlines()

                if not lines:
                    backgrounds += 1

                for line in lines:
                    p = line.split()
                    if len(p) != 5:
                        raise ValueError(f"Invalid label: {label_name}")

                    cls = int(p[0])
                    coords = [float(v) for v in p[1:]]

                    if cls not in range(4):
                        raise ValueError(f"Invalid class: {cls}")

                    if not all(0 <= v <= 1 for v in coords):
                        raise ValueError("Invalid coordinates")

                    if coords[2] <= 0 or coords[3] <= 0:
                        raise ValueError("Invalid box dimensions")

                    boxes[cls] += 1

                if split in ["val", "test"]:
                    eval_hashes[image_name] = hashlib.sha256(
                        z.read(image_name)
                    ).hexdigest()
                    eval_hashes[label_name] = hashlib.sha256(
                        z.read(label_name)
                    ).hexdigest()

            splits[split] = {
                "images": len(images),
                "backgrounds": backgrounds,
                "boxes_per_class": dict(boxes)
            }

    return splits, eval_hashes

v1, v1_eval = read_dataset(ROOT / "data/rdd2022_v1.zip")
v2, v2_eval = read_dataset(ROOT / "data/rdd2022_v2.zip")

if v1_eval != v2_eval:
    raise ValueError("Validation/test differs between versions")

if [v1[s]["images"] for s in ["train","val","test"]] != [300,75,75]:
    raise ValueError("Unexpected v1 sizes")

if [v2[s]["images"] for s in ["train","val","test"]] != [1000,75,75]:
    raise ValueError("Unexpected v2 sizes")

report = {
    "v1": v1,
    "v2": v2,
    "evaluation_unchanged": True
}

output = ROOT / "reports/data_validation.json"
output.parent.mkdir(exist_ok=True)
output.write_text(json.dumps(report, indent=2))

print("Dataset validation passed!")
print(json.dumps(report, indent=2))
'''

(PROJECT / "src/verify_data.py").write_text(script)
print("Validation script created!")


Validation script created!


In [11]:
dvc_yaml = """
stages:
  validate_data:
    cmd: python src/verify_data.py
    deps:
      - src/verify_data.py
      - data/rdd2022_v1.zip
      - data/rdd2022_v2.zip
    outs:
      - reports/data_validation.json
"""

(PROJECT / "dvc.yaml").write_text(dvc_yaml)

print((PROJECT / "dvc.yaml").read_text())


stages:
  validate_data:
    cmd: python src/verify_data.py
    deps:
      - src/verify_data.py
      - data/rdd2022_v1.zip
      - data/rdd2022_v2.zip
    outs:
      - reports/data_validation.json



In [12]:
run("dvc", "repro")

In [13]:
run("dvc", "repro")

In [14]:
run("git", "add", ".")
run(
    "git",
    "-c", "user.name=RoadVision",
    "-c", "user.email=roadvision@example.invalid",
    "commit",
    "-m", "Add reproducible dataset validation pipeline"
)

run("dvc", "push")

In [15]:
BUNDLE_DIR = DRIVE_ROOT / "git_backup"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

bundle_path = BUNDLE_DIR / "roadvision-mlops.bundle"

run(
    "git", "bundle", "create",
    str(bundle_path), "--all"
)

print("Git history backed up:", bundle_path)

Git history backed up: /content/drive/MyDrive/RoadVision/git_backup/roadvision-mlops.bundle


In [16]:

import subprocess
import json

def check(*args):
    result = subprocess.run(
        list(args),
        cwd=PROJECT,
        capture_output=True,
        text=True,
        check=True
    )
    print(f"\n$ {' '.join(args)}")
    print(result.stdout or "(no output)")
    if result.stderr:
        print(result.stderr)
    return result.stdout

# 1. DVC pipeline status
check("dvc", "status")

# 2. DVC remote status
check("dvc", "status", "-c")

# 3. Git commits
check("git", "log", "--oneline", "-3")

# 4. Verify Git backup
check("git", "bundle", "verify", str(bundle_path))

# 5. Read data validation report
report_path = PROJECT / "reports/data_validation.json"
assert report_path.exists(), "Validation report missing!"

report = json.loads(report_path.read_text())

print("\nEvaluation unchanged:",
      report["evaluation_unchanged"])

assert report["evaluation_unchanged"] is True

print("\nPHASE 5 VERIFICATION COMPLETED!")



$ dvc status
Data and pipelines are up to date.


$ dvc status -c
Cache and remote 'drive-storage' are in sync.


$ git log --oneline -3
305e503 Add reproducible dataset validation pipeline
5797596 Track RoadVision v1 and v2 datasets and models with DVC


$ git bundle verify /content/drive/MyDrive/RoadVision/git_backup/roadvision-mlops.bundle
The bundle contains these 2 refs:
305e50358112a96e1d4a2f8cae471896678cd146 refs/heads/main
305e50358112a96e1d4a2f8cae471896678cd146 HEAD
The bundle records a complete history.
The bundle uses this hash algorithm: sha1

/content/drive/MyDrive/RoadVision/git_backup/roadvision-mlops.bundle is okay


Evaluation unchanged: True

PHASE 5 VERIFICATION COMPLETED!
